In [1]:
import os
import socket
import pandas as pd
import pyarrow
import numpy as np

print(socket.gethostname())
os.chdir("/home/u24211510018/workspace/Atlas_WGBS/tissue_cell_type_methylation_density")
print(os.getcwd())

cpu10
/home/u24211510018/workspace/Atlas_WGBS/tissue_cell_type_methylation_density


In [3]:
combined_df = pd.read_parquet("../R_script_data/combined_methylation_density_matrix.parquet")

In [4]:
combined_df.head()

,Adipocytes,Adipocytes.1,Adipocytes.2,Aorta-Endothel,Aorta-Endothel.1,Saphenous-Vein-Endothel,Saphenous-Vein-Endothel.1,Saphenous-Vein-Endothel.2,Kidney-Glomerular-Endothel,Kidney-Glomerular-Endothel.1,...,Colon-Left-Epithelial,Colon-Left-Endocrine,Colon-Left-Endocrine.1,Colon-Left-Epithelial.1,Colon-Left-Epithelial.2,Small-int-Epithelial,Small-int-Epithelial.1,Small-int-Epithelial.2,Small-int-Epithelial.3,Small-int-Endocrine
chr1_10000_10500,61.117,52.833,84.683,50.000,76.800,82.317,82.583,48.100,77.233,74.300,...,85.717,78.750,87.300,89.483,91.250,86.117,92.850,86.200,92.083,86.117
chr1_10500_11000,74.198,60.226,68.778,60.293,64.196,71.074,64.989,57.583,76.109,71.362,...,59.211,62.950,76.596,67.190,59.265,64.470,69.326,67.788,55.567,59.306
chr1_11000_11500,5.556,NaN,NaN,2.778,NaN,NaN,NaN,NaN,8.333,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
chr1_12500_13000,7.143,NaN,35.714,NaN,NaN,NaN,50.000,35.714,NaN,NaN,...,NaN,28.571,NaN,7.143,NaN,7.143,NaN,NaN,NaN,35.714
chr1_13000_13500,63.120,62.980,59.520,64.200,NaN,55.560,42.342,48.700,76.200,58.620,...,65.726,51.120,51.440,60.892,60.260,52.974,47.540,NaN,57.720,59.180


In [5]:
print(combined_df.shape)
print(combined_df.columns)
print(combined_df.index)

(5126737, 207)
Index(['Adipocytes', 'Adipocytes.1', 'Adipocytes.2', 'Aorta-Endothel',
       'Aorta-Endothel.1', 'Saphenous-Vein-Endothel',
       'Saphenous-Vein-Endothel.1', 'Saphenous-Vein-Endothel.2',
       'Kidney-Glomerular-Endothel', 'Kidney-Glomerular-Endothel.1',
       ...
       'Colon-Left-Epithelial', 'Colon-Left-Endocrine',
       'Colon-Left-Endocrine.1', 'Colon-Left-Epithelial.1',
       'Colon-Left-Epithelial.2', 'Small-int-Epithelial',
       'Small-int-Epithelial.1', 'Small-int-Epithelial.2',
       'Small-int-Epithelial.3', 'Small-int-Endocrine'],
      dtype='object', length=207)
Index(['chr1_10000_10500', 'chr1_10500_11000', 'chr1_11000_11500',
       'chr1_12500_13000', 'chr1_13000_13500', 'chr1_13500_14000',
       'chr1_14000_14500', 'chr1_14500_15000', 'chr1_15000_15500',
       'chr1_15500_16000',
       ...
       'chr5_71232500_71233000', 'chr6_59670500_59671000',
       'chr7_61668000_61668500', 'chr9_61907500_61908000',
       'chr9_62010000_62010500', '

# combine with megakaryocyte

In [6]:
import os
import pandas as pd

def read_methylation_bed(file_path):
    # 读取文件
    df = pd.read_csv(
        file_path,
        sep="\t",
        header=None,
        names=["chr", "start", "end", "value"]
    )
    
    # 合成 region_id
    df["region_id"] = (
        df["chr"].astype(str) + "_" +
        df["start"].astype(str) + "_" +
        df["end"].astype(str)
    )
    
    # 转换成 series，index 为 region_id，value 为 methylation density
    s = pd.Series(df["value"].values, index=df["region_id"].values, name=os.path.basename(file_path))
    return s

# 文件路径
megakaryocyte_files = [
    "/home/u24211510018/workspace/Atlas_WGBS/Megakaryocyte/GSE87196_RAW/MK_D1_sample/MK_D1.CpG_dyad.methylation_density.bed",
    "/home/u24211510018/workspace/Atlas_WGBS/Megakaryocyte/GSE87196_RAW/MK_D2_sample/MK_D2.CpG_dyad.methylation_density.bed",
    "/home/u24211510018/workspace/Atlas_WGBS/Megakaryocyte/GSE87196_RAW/MK_D3_sample/MK_D3.CpG_dyad.methylation_density.bed"
]

# 将三个文件读入成 pandas Series
megakaryocyte_series_dict = {}
for idx, file_path in enumerate(megakaryocyte_files, start=1):
    sample_name = f"Megakaryocytes.{idx}"
    megakaryocyte_series_dict[sample_name] = read_methylation_bed(file_path)

# 打印检查
for name, series in megakaryocyte_series_dict.items():
    print(name, series.head())


Megakaryocytes.1 chr1_10000_10500    33.333
chr1_10500_11000    39.189
chr1_12500_13000    21.429
chr1_13000_13500    60.000
chr1_13500_14000     0.000
Name: MK_D1.CpG_dyad.methylation_density.bed, dtype: float64
Megakaryocytes.2 chr1_10000_10500    16.667
chr1_10500_11000    20.732
chr1_13000_13500    33.340
chr1_13500_14000     0.000
chr1_14500_15000    43.750
Name: MK_D2.CpG_dyad.methylation_density.bed, dtype: float64
Megakaryocytes.3 chr1_10000_10500    50.000
chr1_10500_11000     8.537
chr1_13000_13500    60.000
chr1_13500_14000     0.000
chr1_14500_15000    15.625
Name: MK_D3.CpG_dyad.methylation_density.bed, dtype: float64


In [7]:
# 把 megakaryocyte 数据合并到 combined_df
for sample_name, sample_series in megakaryocyte_series_dict.items():
    # 合并数据
    combined_df[sample_name] = sample_series

# 检查合并后的结果
print(combined_df.shape)
print(combined_df.head())

(5126737, 210)
                  Adipocytes  Adipocytes.1  Adipocytes.2  Aorta-Endothel  \
chr1_10000_10500      61.117        52.833        84.683          50.000   
chr1_10500_11000      74.198        60.226        68.778          60.293   
chr1_11000_11500       5.556           NaN           NaN           2.778   
chr1_12500_13000       7.143           NaN        35.714             NaN   
chr1_13000_13500      63.120        62.980        59.520          64.200   

                  Aorta-Endothel.1  Saphenous-Vein-Endothel  \
chr1_10000_10500            76.800                   82.317   
chr1_10500_11000            64.196                   71.074   
chr1_11000_11500               NaN                      NaN   
chr1_12500_13000               NaN                      NaN   
chr1_13000_13500               NaN                   55.560   

                  Saphenous-Vein-Endothel.1  Saphenous-Vein-Endothel.2  \
chr1_10000_10500                     82.583                     48.100   
c

In [8]:
# reremove rows with NA values
combined__clean_df = combined_df.dropna()

In [9]:
print(combined__clean_df.shape)
combined__clean_df.head()

(3456189, 210)


,Adipocytes,Adipocytes.1,Adipocytes.2,Aorta-Endothel,Aorta-Endothel.1,Saphenous-Vein-Endothel,Saphenous-Vein-Endothel.1,Saphenous-Vein-Endothel.2,Kidney-Glomerular-Endothel,Kidney-Glomerular-Endothel.1,...,Colon-Left-Epithelial.1,Colon-Left-Epithelial.2,Small-int-Epithelial,Small-int-Epithelial.1,Small-int-Epithelial.2,Small-int-Epithelial.3,Small-int-Endocrine,Megakaryocytes.1,Megakaryocytes.2,Megakaryocytes.3
chr1_10000_10500,61.117,52.833,84.683,50.000,76.800,82.317,82.583,48.100,77.233,74.300,...,89.483,91.250,86.117,92.850,86.200,92.083,86.117,33.333,16.667,50.000
chr1_10500_11000,74.198,60.226,68.778,60.293,64.196,71.074,64.989,57.583,76.109,71.362,...,67.190,59.265,64.470,69.326,67.788,55.567,59.306,39.189,20.732,8.537
chr1_14500_15000,83.262,60.881,70.263,85.431,61.300,78.281,87.287,86.619,79.350,77.056,...,92.406,71.306,82.056,78.694,54.425,66.881,80.575,35.419,43.750,15.625
chr1_16500_17000,51.060,51.407,50.167,53.787,38.707,52.793,54.153,49.667,54.673,56.247,...,63.140,48.933,48.520,51.013,43.460,48.460,44.887,33.333,20.000,33.333
chr1_17000_17500,90.467,91.417,89.100,94.567,93.667,93.817,95.400,94.883,94.600,92.817,...,93.350,94.250,96.467,83.250,94.683,92.517,93.050,93.617,100.000,75.000


In [10]:
print(combined__clean_df.columns)
print(combined__clean_df.index)

Index(['Adipocytes', 'Adipocytes.1', 'Adipocytes.2', 'Aorta-Endothel',
       'Aorta-Endothel.1', 'Saphenous-Vein-Endothel',
       'Saphenous-Vein-Endothel.1', 'Saphenous-Vein-Endothel.2',
       'Kidney-Glomerular-Endothel', 'Kidney-Glomerular-Endothel.1',
       ...
       'Colon-Left-Epithelial.1', 'Colon-Left-Epithelial.2',
       'Small-int-Epithelial', 'Small-int-Epithelial.1',
       'Small-int-Epithelial.2', 'Small-int-Epithelial.3',
       'Small-int-Endocrine', 'Megakaryocytes.1', 'Megakaryocytes.2',
       'Megakaryocytes.3'],
      dtype='object', length=210)
Index(['chr1_10000_10500', 'chr1_10500_11000', 'chr1_14500_15000',
       'chr1_16500_17000', 'chr1_17000_17500', 'chr1_17500_18000',
       'chr1_28500_29000', 'chr1_102500_103000', 'chr1_103500_104000',
       'chr1_104000_104500',
       ...
       'chr9_138214500_138215000', 'chr9_138216500_138217000',
       'chr9_138217000_138217500', 'chr9_138217500_138218000',
       'chr9_138218500_138219000', 'chr9_138219000

In [85]:
# 将 combined_clean_mean_df 保存为 Parquet 文件
# combined__clean_df.to_parquet("../R_script_data/combined__MK_clean_df.parquet", engine='pyarrow')

In [12]:
import pandas as pd

# 去掉重复样本后缀，例如 Adipocytes.1 -> Adipocytes
group_cols = combined__clean_df.columns.to_series().str.replace(r"\.\d+$", "", regex=True)

# 按组名合并，按行取平均值
combined_clean_mean_df = combined__clean_df.T.groupby(group_cols).mean().T

print(combined_clean_mean_df.shape)

(3456189, 83)


In [13]:
combined_clean_mean_df.head()

,Adipocytes,Aorta-Endothel,Aorta-Smooth-Muscle,Bladder-Epithelial,Bladder-Smooth-Muscle,Blood-B,Blood-B-Mem,Blood-Granulocytes,Blood-Monocytes,Blood-NK,...,Prostate-Smooth-Muscle,Saphenous-Vein-Endothel,Skeletal-Muscle,Small-int-Endocrine,Small-int-Epithelial,Thyroid-Epithelial,Tongue-Epithelial,Tongue_base-Epithelial,Tonsil-Palatine-Epithelial,Tonsil-Pharyngeal-Epithelial
chr1_10000_10500,66.211000,63.4000,65.800,72.7134,43.333,82.000000,84.225,75.844667,74.888667,80.194333,...,79.083,71.000000,70.9500,86.117,89.31250,71.405667,78.044667,60.283,76.383333,72.8750
chr1_10500_11000,67.734000,62.2445,65.749,67.1190,77.920,70.147667,44.870,64.061667,62.626333,58.586000,...,62.396,64.548667,58.5885,59.306,64.28775,60.647000,79.430000,63.529,75.941333,59.4905
chr1_14500_15000,71.468667,73.3655,72.319,75.1838,83.438,68.585333,67.428,71.687667,61.556333,59.681333,...,76.481,84.062333,70.2720,80.575,70.51400,57.004000,83.498000,75.150,81.160667,81.3560
chr1_16500_17000,50.878000,46.2470,34.020,46.7654,45.227,53.240000,43.810,51.911000,50.602333,50.877667,...,51.800,52.204333,48.5465,44.887,47.86325,53.217667,46.062000,48.473,47.733333,48.6100
chr1_17000_17500,90.328000,94.1170,87.017,95.5168,86.267,95.094333,94.858,93.627667,94.583333,94.627667,...,91.300,94.700000,90.7250,93.050,91.72925,96.194333,89.555667,97.050,92.278000,90.4665


In [14]:
print(combined_clean_mean_df.columns)
print(combined_clean_mean_df.index)

Index(['Adipocytes', 'Aorta-Endothel', 'Aorta-Smooth-Muscle',
       'Bladder-Epithelial', 'Bladder-Smooth-Muscle', 'Blood-B', 'Blood-B-Mem',
       'Blood-Granulocytes', 'Blood-Monocytes', 'Blood-NK', 'Blood-T-CD3',
       'Blood-T-CD4', 'Blood-T-CD8', 'Blood-T-CenMem-CD4', 'Blood-T-Eff-CD8',
       'Blood-T-EffMem-CD4', 'Blood-T-EffMem-CD8', 'Blood-T-Naive-CD4',
       'Blood-T-Naive-CD8', 'Bone-Osteoblasts',
       'Bone_marrow-Erythrocyte_progenitors', 'Breast-Basal-Epithelial',
       'Breast-Luminal-Epithelial', 'Cerebellum-Neuron', 'Colon-Fibroblasts',
       'Colon-Left-Endocrine', 'Colon-Left-Epithelial', 'Colon-Macrophages',
       'Colon-Right-Endocrine', 'Colon-Right-Epithelial',
       'Coronary-Artery-Smooth-Muscle', 'Cortex-Neuron', 'Dermal-Fibroblasts',
       'Endometrium-Epithelial', 'Epidermal-Keratinocytes',
       'Esophagus-Epithelial', 'Fallopian-Epithelial',
       'Gallbladder-Epithelial', 'Gastric-antrum-Endocrine',
       'Gastric-antrum-Epithelial', 'Gastric

In [61]:
# 将 combined_clean_mean_df 保存为 Parquet 文件
# combined_clean_mean_df.to_parquet("../R_script_data/combined_clean_mean_df.parquet", engine='pyarrow')

# read combined date

In [15]:
combined_clean_mean_df = pd.read_parquet("../R_script_data/combined_clean_mean_df.parquet")

# construct ref map for deconvolution

In [16]:
ref_df = combined_clean_mean_df.copy()

print("原始参考矩阵大小：", ref_df.shape)
ref_df.head()

原始参考矩阵大小： (3456189, 83)


,Adipocytes,Aorta-Endothel,Aorta-Smooth-Muscle,Bladder-Epithelial,Bladder-Smooth-Muscle,Blood-B,Blood-B-Mem,Blood-Granulocytes,Blood-Monocytes,Blood-NK,...,Prostate-Smooth-Muscle,Saphenous-Vein-Endothel,Skeletal-Muscle,Small-int-Endocrine,Small-int-Epithelial,Thyroid-Epithelial,Tongue-Epithelial,Tongue_base-Epithelial,Tonsil-Palatine-Epithelial,Tonsil-Pharyngeal-Epithelial
chr1_10000_10500,66.211000,63.4000,65.800,72.7134,43.333,82.000000,84.225,75.844667,74.888667,80.194333,...,79.083,71.000000,70.9500,86.117,89.31250,71.405667,78.044667,60.283,76.383333,72.8750
chr1_10500_11000,67.734000,62.2445,65.749,67.1190,77.920,70.147667,44.870,64.061667,62.626333,58.586000,...,62.396,64.548667,58.5885,59.306,64.28775,60.647000,79.430000,63.529,75.941333,59.4905
chr1_14500_15000,71.468667,73.3655,72.319,75.1838,83.438,68.585333,67.428,71.687667,61.556333,59.681333,...,76.481,84.062333,70.2720,80.575,70.51400,57.004000,83.498000,75.150,81.160667,81.3560
chr1_16500_17000,50.878000,46.2470,34.020,46.7654,45.227,53.240000,43.810,51.911000,50.602333,50.877667,...,51.800,52.204333,48.5465,44.887,47.86325,53.217667,46.062000,48.473,47.733333,48.6100
chr1_17000_17500,90.328000,94.1170,87.017,95.5168,86.267,95.094333,94.858,93.627667,94.583333,94.627667,...,91.300,94.700000,90.7250,93.050,91.72925,96.194333,89.555667,97.050,92.278000,90.4665


In [17]:
print(ref_df.index)
print(ref_df.columns)

Index(['chr1_10000_10500', 'chr1_10500_11000', 'chr1_14500_15000',
       'chr1_16500_17000', 'chr1_17000_17500', 'chr1_17500_18000',
       'chr1_28500_29000', 'chr1_102500_103000', 'chr1_103500_104000',
       'chr1_104000_104500',
       ...
       'chr9_138214500_138215000', 'chr9_138216500_138217000',
       'chr9_138217000_138217500', 'chr9_138217500_138218000',
       'chr9_138218500_138219000', 'chr9_138219000_138219500',
       'chr9_138220000_138220500', 'chr9_138220500_138221000',
       'chr9_138232000_138232500', 'chr9_138233000_138233500'],
      dtype='object', length=3456189)
Index(['Adipocytes', 'Aorta-Endothel', 'Aorta-Smooth-Muscle',
       'Bladder-Epithelial', 'Bladder-Smooth-Muscle', 'Blood-B', 'Blood-B-Mem',
       'Blood-Granulocytes', 'Blood-Monocytes', 'Blood-NK', 'Blood-T-CD3',
       'Blood-T-CD4', 'Blood-T-CD8', 'Blood-T-CenMem-CD4', 'Blood-T-Eff-CD8',
       'Blood-T-EffMem-CD4', 'Blood-T-EffMem-CD8', 'Blood-T-Naive-CD4',
       'Blood-T-Naive-CD8', 'Bone-

In [18]:
cpg_bed = pd.read_csv(
    "../hg38_500bp_blocks_with_CpG_dyad.bed",
    sep="\t",
    header=None,
    names=["chr", "start", "end", "block_id", "cpg_count"]
)

print(cpg_bed.head())
print(cpg_bed.shape)

    chr  start    end  block_id  cpg_count
0  chr1  10000  10500  block_21          6
1  chr1  10500  11000  block_22         82
2  chr1  11000  11500  block_23         36
3  chr1  11500  12000  block_24          9
4  chr1  12000  12500  block_25          8
(5150517, 5)


In [19]:
cpg_bed["region_id"] = (
    cpg_bed["chr"].astype(str) + "_" +
    cpg_bed["start"].astype(str) + "_" +
    cpg_bed["end"].astype(str)
)

print(cpg_bed[["region_id", "cpg_count"]].head())
cpg_bed.head()

          region_id  cpg_count
0  chr1_10000_10500          6
1  chr1_10500_11000         82
2  chr1_11000_11500         36
3  chr1_11500_12000          9
4  chr1_12000_12500          8


,chr,start,end,block_id,cpg_count,region_id
0,chr1,10000,10500,block_21,6,chr1_10000_10500
1,chr1,10500,11000,block_22,82,chr1_10500_11000
2,chr1,11000,11500,block_23,36,chr1_11000_11500
3,chr1,11500,12000,block_24,9,chr1_11500_12000
4,chr1,12000,12500,block_25,8,chr1_12000_12500


In [20]:
cpg_info = cpg_bed[["region_id", "cpg_count"]].copy()
cpg_info = cpg_info[cpg_info["region_id"].isin(ref_df.index)].copy()

print("参考矩阵中匹配到 CpG 信息的 block 数：", cpg_info.shape[0])

参考矩阵中匹配到 CpG 信息的 block 数： 3456189


In [21]:
cpg_info.head()

,region_id,cpg_count
0,chr1_10000_10500,6
1,chr1_10500_11000,82
9,chr1_14500_15000,16
13,chr1_16500_17000,15
14,chr1_17000_17500,6


In [22]:
min_cpg_count = 5

cpg_pass_blocks = cpg_info.loc[cpg_info["cpg_count"] >= min_cpg_count, "region_id"]

ref_df_cpg = ref_df.loc[ref_df.index.intersection(cpg_pass_blocks)].copy()

print("原始 block 数：", ref_df.shape[0])
print("CpG 过滤后 block 数：", ref_df_cpg.shape[0])

原始 block 数： 3456189
CpG 过滤后 block 数： 1889819


In [23]:
ref_df_cpg.head()

,Adipocytes,Aorta-Endothel,Aorta-Smooth-Muscle,Bladder-Epithelial,Bladder-Smooth-Muscle,Blood-B,Blood-B-Mem,Blood-Granulocytes,Blood-Monocytes,Blood-NK,...,Prostate-Smooth-Muscle,Saphenous-Vein-Endothel,Skeletal-Muscle,Small-int-Endocrine,Small-int-Epithelial,Thyroid-Epithelial,Tongue-Epithelial,Tongue_base-Epithelial,Tonsil-Palatine-Epithelial,Tonsil-Pharyngeal-Epithelial
chr1_10000_10500,66.211000,63.4000,65.800,72.7134,43.333,82.000000,84.225,75.844667,74.888667,80.194333,...,79.083,71.000000,70.9500,86.117,89.31250,71.405667,78.044667,60.283,76.383333,72.8750
chr1_10500_11000,67.734000,62.2445,65.749,67.1190,77.920,70.147667,44.870,64.061667,62.626333,58.586000,...,62.396,64.548667,58.5885,59.306,64.28775,60.647000,79.430000,63.529,75.941333,59.4905
chr1_14500_15000,71.468667,73.3655,72.319,75.1838,83.438,68.585333,67.428,71.687667,61.556333,59.681333,...,76.481,84.062333,70.2720,80.575,70.51400,57.004000,83.498000,75.150,81.160667,81.3560
chr1_16500_17000,50.878000,46.2470,34.020,46.7654,45.227,53.240000,43.810,51.911000,50.602333,50.877667,...,51.800,52.204333,48.5465,44.887,47.86325,53.217667,46.062000,48.473,47.733333,48.6100
chr1_17000_17500,90.328000,94.1170,87.017,95.5168,86.267,95.094333,94.858,93.627667,94.583333,94.627667,...,91.300,94.700000,90.7250,93.050,91.72925,96.194333,89.555667,97.050,92.278000,90.4665


In [24]:
block_mean = ref_df_cpg.mean(axis=1)
block_sd = ref_df_cpg.std(axis=1)

# CV = SD / mean (Coefficient of Variation)
block_cv = block_sd / block_mean.replace(0, np.nan)

block_stats = pd.DataFrame({
    "mean": block_mean,
    "sd": block_sd,
    "cv": block_cv
})

print(block_stats.head())
print(block_stats.shape)


                       mean         sd        cv
chr1_10000_10500  73.983167   9.075298  0.122667
chr1_10500_11000  65.179030   8.634113  0.132468
chr1_14500_15000  72.276427  11.625469  0.160847
chr1_16500_17000  50.617217   5.002716  0.098834
chr1_17000_17500  93.011119   2.212540  0.023788
(1889819, 3)


In [25]:
cv_threshold = 0.35

cv_pass_blocks = block_stats.index[block_stats["cv"] >= cv_threshold]

filtered_df = ref_df_cpg.loc[cv_pass_blocks].copy()

print("CpG 过滤后 block 数：", ref_df_cpg.shape[0])
print("CpG 过滤 + CV 过滤后 block 数：", filtered_df.shape[0])

CpG 过滤后 block 数： 1889819
CpG 过滤 + CV 过滤后 block 数： 158392


In [26]:
print(block_stats["cv"].describe())

count    1.889819e+06
mean     1.630364e-01
std      1.445568e-01
min      9.558799e-03
25%      7.799462e-02
50%      1.201238e-01
75%      1.908076e-01
max      2.462603e+00
Name: cv, dtype: float64


In [27]:
filtered_df.head()

,Adipocytes,Aorta-Endothel,Aorta-Smooth-Muscle,Bladder-Epithelial,Bladder-Smooth-Muscle,Blood-B,Blood-B-Mem,Blood-Granulocytes,Blood-Monocytes,Blood-NK,...,Prostate-Smooth-Muscle,Saphenous-Vein-Endothel,Skeletal-Muscle,Small-int-Endocrine,Small-int-Epithelial,Thyroid-Epithelial,Tongue-Epithelial,Tongue_base-Epithelial,Tonsil-Palatine-Epithelial,Tonsil-Pharyngeal-Epithelial
chr1_28500_29000,0.689667,0.3450,0.000,0.9192,0.000,2.401000,1.7240,2.706667,3.706667,2.463333,...,0.000,2.873333,0.0000,0.000,1.29300,0.287333,0.183333,0.000,1.478000,0.0000
chr1_102500_103000,59.061000,57.3085,36.567,52.1434,29.667,67.805667,63.0750,82.505333,75.216667,66.549667,...,36.933,74.116667,54.1835,5.388,2.99825,72.522333,14.489333,10.638,21.417667,43.6665
chr1_103500_104000,62.017000,41.6625,30.950,55.8802,42.150,26.391333,45.5745,10.316333,15.283000,31.666667,...,75.388,56.079333,50.4875,57.612,60.57800,53.433333,72.554000,82.713,65.258667,11.2250
chr1_191500_192000,37.903333,44.2210,51.864,76.5098,50.000,68.638333,67.8355,21.513000,43.701667,37.835333,...,62.479,29.569667,53.0680,73.343,80.12300,82.626333,82.016667,68.036,71.390667,70.4285
chr1_629500_630000,7.802000,8.9505,10.580,2.9256,3.878,8.741000,12.3535,16.647333,10.765000,15.347667,...,18.742,10.903000,32.5240,34.309,17.90150,16.224667,12.594667,7.594,17.405000,15.4855


In [28]:
tissues = filtered_df.columns.tolist()

marker_result = {}

for tissue in tissues:
    this_values = filtered_df[tissue]
    
    other_tissues = [x for x in tissues if x != tissue]
    other_df = filtered_df[other_tissues]
    
    other_max = other_df.max(axis=1)
    other_min = other_df.min(axis=1)
    other_mean = other_df.mean(axis=1)
    other_sd = other_df.std(axis=1)
    
    hyper_score = this_values - other_max
    hypo_score = other_min - this_values
    
    hyper_zscore = (this_values - other_mean) / other_sd.replace(0, np.nan)
    hypo_zscore = (other_mean - this_values) / other_sd.replace(0, np.nan)
    
    tissue_df = pd.DataFrame({
        "block": filtered_df.index,
        "tissue": tissue,
        "value": this_values.values,
        "other_max": other_max.values,
        "other_min": other_min.values,
        "other_mean": other_mean.values,
        "other_sd": other_sd.values,
        "hyper_score": hyper_score.values,
        "hypo_score": hypo_score.values,
        "hyper_zscore": hyper_zscore.values,
        "hypo_zscore": hypo_zscore.values
    })
    
    marker_result[tissue] = tissue_df

marker_result[tissues[0]].head()

,block,tissue,value,other_max,other_min,other_mean,other_sd,hyper_score,hypo_score,hyper_zscore,hypo_zscore
0,chr1_28500_29000,Adipocytes,0.689667,4.214,0.000000,1.185749,1.221678,-3.524333,-0.689667,-0.406066,0.406066
1,chr1_102500_103000,Adipocytes,59.061000,95.583,2.998250,47.121713,24.647787,-36.522000,-56.062750,0.484396,-0.484396
2,chr1_103500_104000,Adipocytes,62.017000,92.500,10.258333,54.445038,20.618216,-30.483000,-51.758667,0.367246,-0.367246
3,chr1_191500_192000,Adipocytes,37.903333,87.657,8.560000,59.486769,21.024066,-49.753667,-29.343333,-1.026606,1.026606
4,chr1_629500_630000,Adipocytes,7.802000,56.695,0.099000,16.738831,10.076207,-48.893000,-7.703000,-0.886924,0.886924


# Do not distinguish between hyper and hypo

In [29]:
top_n = 50

top_marker_tables = []

for tissue in tissues:
    tissue_df = marker_result[tissue].copy()
    
    # hyper 候选
    hyper_df = tissue_df[tissue_df["hyper_score"] > 0].copy()
    hyper_df["marker_type"] = "hyper"
    hyper_df["score_abs"] = hyper_df["hyper_score"].abs()
    
    # hypo 候选
    hypo_df = tissue_df[tissue_df["hypo_score"] > 0].copy()
    hypo_df["marker_type"] = "hypo"
    hypo_df["score_abs"] = hypo_df["hypo_score"].abs()
    
    # 合并
    merged_df = pd.concat([hyper_df, hypo_df], axis=0, ignore_index=True)
    
    # 按绝对值排序，取前 top_n
    merged_df = merged_df.sort_values("score_abs", ascending=False).head(top_n)
    
    top_marker_tables.append(merged_df)

all_markers_df = pd.concat(top_marker_tables, axis=0, ignore_index=True)

print(all_markers_df.shape)
all_markers_df.head()

(4145, 13)


,block,tissue,value,other_max,other_min,other_mean,other_sd,hyper_score,hypo_score,hyper_zscore,hypo_zscore,marker_type,score_abs
0,chr18_79358000_79358500,Adipocytes,67.405333,52.244667,5.555667,24.783869,10.968173,15.160667,-61.849667,3.885922,-3.885922,hyper,15.160667
1,chr5_181053500_181054000,Adipocytes,21.302000,9.694000,0.416667,4.679515,1.652703,11.608000,-20.885333,10.057756,-10.057756,hyper,11.608000
2,chr8_127391500_127392000,Adipocytes,16.136667,98.420000,23.204000,62.650715,24.096067,-82.283333,7.067333,-1.930359,1.930359,hypo,7.067333
3,chr22_38317000_38317500,Adipocytes,11.347667,6.769333,0.625000,2.975548,1.033991,4.578333,-10.722667,8.096899,-8.096899,hyper,4.578333
4,chr1_153774500_153775000,Adipocytes,13.373667,92.433500,17.688000,49.576527,19.628529,-79.059833,4.314333,-1.844400,1.844400,hypo,4.314333


In [30]:
pd.set_option('display.max_rows', None)
marker_count_table = (
    all_markers_df
    .groupby(["tissue", "marker_type"])
    .size()
    .reset_index(name="count")
)

print(marker_count_table)

                                  tissue marker_type  count
0                             Adipocytes       hyper     15
1                             Adipocytes        hypo     30
2                         Aorta-Endothel       hyper     32
3                         Aorta-Endothel        hypo     18
4                    Aorta-Smooth-Muscle       hyper     31
5                    Aorta-Smooth-Muscle        hypo     19
6                     Bladder-Epithelial       hyper     50
7                  Bladder-Smooth-Muscle       hyper     27
8                  Bladder-Smooth-Muscle        hypo     23
9                                Blood-B       hyper     36
10                               Blood-B        hypo     14
11                           Blood-B-Mem       hyper     14
12                           Blood-B-Mem        hypo     36
13                    Blood-Granulocytes       hyper     11
14                    Blood-Granulocytes        hypo     39
15                       Blood-Monocytes

In [31]:
final_marker_blocks = all_markers_df["block"].drop_duplicates().tolist()

print("最终去重后的 marker block 数：", len(final_marker_blocks))
final_marker_blocks[:10]

最终去重后的 marker block 数： 4138


['chr18_79358000_79358500',
 'chr5_181053500_181054000',
 'chr8_127391500_127392000',
 'chr22_38317000_38317500',
 'chr1_153774500_153775000',
 'chr4_169059500_169060000',
 'chr11_849500_850000',
 'chr11_71096000_71096500',
 'chr10_79435500_79436000',
 'chr2_176153000_176153500']

In [32]:
all_markers_df.head()

,block,tissue,value,other_max,other_min,other_mean,other_sd,hyper_score,hypo_score,hyper_zscore,hypo_zscore,marker_type,score_abs
0,chr18_79358000_79358500,Adipocytes,67.405333,52.244667,5.555667,24.783869,10.968173,15.160667,-61.849667,3.885922,-3.885922,hyper,15.160667
1,chr5_181053500_181054000,Adipocytes,21.302000,9.694000,0.416667,4.679515,1.652703,11.608000,-20.885333,10.057756,-10.057756,hyper,11.608000
2,chr8_127391500_127392000,Adipocytes,16.136667,98.420000,23.204000,62.650715,24.096067,-82.283333,7.067333,-1.930359,1.930359,hypo,7.067333
3,chr22_38317000_38317500,Adipocytes,11.347667,6.769333,0.625000,2.975548,1.033991,4.578333,-10.722667,8.096899,-8.096899,hyper,4.578333
4,chr1_153774500_153775000,Adipocytes,13.373667,92.433500,17.688000,49.576527,19.628529,-79.059833,4.314333,-1.844400,1.844400,hypo,4.314333


In [45]:
final_marker_annotation = (
    all_markers_df
    .groupby("block")
    .agg({
        "tissue": lambda x: ",".join(sorted(set(x))),
        "marker_type": lambda x: ",".join(sorted(set(x)))
    })
    .reset_index()
)

print(final_marker_annotation.head())
print(final_marker_annotation.shape)

                       block                        tissue marker_type
0  chr10_100342000_100342500                 Cortex-Neuron       hyper
1  chr10_100659500_100660000  Kidney-Glomerular-Epithelial       hyper
2  chr10_100881000_100881500          Small-int-Epithelial        hypo
3  chr10_101113000_101113500       Breast-Basal-Epithelial       hyper
4  chr10_101123000_101123500             Liver-Hepatocytes       hyper
(4138, 3)


In [34]:
# 去重之后的 block: final_marker_blocks
reference_map_df = filtered_df.loc[final_marker_blocks].copy()

print(reference_map_df.shape)
reference_map_df.head()

(4138, 83)


,Adipocytes,Aorta-Endothel,Aorta-Smooth-Muscle,Bladder-Epithelial,Bladder-Smooth-Muscle,Blood-B,Blood-B-Mem,Blood-Granulocytes,Blood-Monocytes,Blood-NK,...,Prostate-Smooth-Muscle,Saphenous-Vein-Endothel,Skeletal-Muscle,Small-int-Endocrine,Small-int-Epithelial,Thyroid-Epithelial,Tongue-Epithelial,Tongue_base-Epithelial,Tonsil-Palatine-Epithelial,Tonsil-Pharyngeal-Epithelial
chr18_79358000_79358500,67.405333,20.8330,27.083,29.0000,8.333,22.483333,15.2750,15.278000,26.666667,18.027667,...,11.117,12.039000,17.2250,11.900,24.98750,23.922000,19.283333,27.783,9.227667,34.3750
chr5_181053500_181054000,21.302000,5.2405,3.851,5.5152,4.864,6.222333,4.3410,4.790667,4.386000,5.502333,...,4.701,4.660000,2.9145,3.539,3.40050,3.505667,4.183333,3.838,3.517667,2.3180
chr8_127391500_127392000,16.136667,46.1000,31.464,46.8748,24.076,80.840000,81.1000,43.970000,52.857333,82.380000,...,27.952,37.666000,39.3490,94.880,92.00000,88.953333,35.123333,34.226,36.710667,39.3190
chr22_38317000_38317500,11.347667,2.3370,4.028,3.5728,2.410,4.122667,2.1875,3.014000,2.812667,3.956000,...,1.850,6.769333,3.0475,1.701,2.89375,1.205000,3.584000,3.333,2.414000,3.4275
chr1_153774500_153775000,13.373667,26.5775,24.642,41.6622,22.517,76.211000,79.6250,44.950667,56.403333,53.824000,...,23.938,25.038333,50.7345,52.332,51.48400,59.358667,28.226667,26.417,29.098000,36.3525


In [35]:
# 左连接，即给final_marker_annotation增加一列 CpG count
final_marker_annotation = final_marker_annotation.merge(
    cpg_info,
    left_on="block",
    right_on="region_id",
    how="left"
)

final_marker_annotation = final_marker_annotation.drop(columns=["region_id"])

print(final_marker_annotation.head())

                       block                        tissue marker_type  \
0  chr10_100342000_100342500                 Cortex-Neuron       hyper   
1  chr10_100659500_100660000  Kidney-Glomerular-Epithelial       hyper   
2  chr10_100881000_100881500          Small-int-Epithelial        hypo   
3  chr10_101113000_101113500       Breast-Basal-Epithelial       hyper   
4  chr10_101123000_101123500             Liver-Hepatocytes       hyper   

   cpg_count  
0          5  
1         32  
2          5  
3          7  
4         28  


In [36]:
print(reference_map_df.shape)
reference_map_df.head()

(4138, 83)


,Adipocytes,Aorta-Endothel,Aorta-Smooth-Muscle,Bladder-Epithelial,Bladder-Smooth-Muscle,Blood-B,Blood-B-Mem,Blood-Granulocytes,Blood-Monocytes,Blood-NK,...,Prostate-Smooth-Muscle,Saphenous-Vein-Endothel,Skeletal-Muscle,Small-int-Endocrine,Small-int-Epithelial,Thyroid-Epithelial,Tongue-Epithelial,Tongue_base-Epithelial,Tonsil-Palatine-Epithelial,Tonsil-Pharyngeal-Epithelial
chr18_79358000_79358500,67.405333,20.8330,27.083,29.0000,8.333,22.483333,15.2750,15.278000,26.666667,18.027667,...,11.117,12.039000,17.2250,11.900,24.98750,23.922000,19.283333,27.783,9.227667,34.3750
chr5_181053500_181054000,21.302000,5.2405,3.851,5.5152,4.864,6.222333,4.3410,4.790667,4.386000,5.502333,...,4.701,4.660000,2.9145,3.539,3.40050,3.505667,4.183333,3.838,3.517667,2.3180
chr8_127391500_127392000,16.136667,46.1000,31.464,46.8748,24.076,80.840000,81.1000,43.970000,52.857333,82.380000,...,27.952,37.666000,39.3490,94.880,92.00000,88.953333,35.123333,34.226,36.710667,39.3190
chr22_38317000_38317500,11.347667,2.3370,4.028,3.5728,2.410,4.122667,2.1875,3.014000,2.812667,3.956000,...,1.850,6.769333,3.0475,1.701,2.89375,1.205000,3.584000,3.333,2.414000,3.4275
chr1_153774500_153775000,13.373667,26.5775,24.642,41.6622,22.517,76.211000,79.6250,44.950667,56.403333,53.824000,...,23.938,25.038333,50.7345,52.332,51.48400,59.358667,28.226667,26.417,29.098000,36.3525


In [54]:
print(final_marker_annotation.head())
multi_tissue = final_marker_annotation[
    final_marker_annotation["tissue"].str.contains(",")
]

multi_tissue.head()

                       block                        tissue marker_type
0  chr10_100342000_100342500                 Cortex-Neuron       hyper
1  chr10_100659500_100660000  Kidney-Glomerular-Epithelial       hyper
2  chr10_100881000_100881500          Small-int-Epithelial        hypo
3  chr10_101113000_101113500       Breast-Basal-Epithelial       hyper
4  chr10_101123000_101123500             Liver-Hepatocytes       hyper


,block,tissue,marker_type
67,chr10_129960500_129961000,"Adipocytes,Prostate-Smooth-Muscle","hyper,hypo"
69,chr10_129962000_129962500,"Adipocytes,Prostate-Smooth-Muscle","hyper,hypo"
1498,chr18_79358000_79358500,"Adipocytes,Pancreas-Endothel","hyper,hypo"
1714,chr19_835500_836000,"Blood-Granulocytes,Blood-T-CD4","hyper,hypo"
2235,chr20_9506500_9507000,"Adipocytes,Endometrium-Epithelial","hyper,hypo"


In [44]:
final_marker_annotation.shape

(4138, 4)

# save the reference map and marker annotation

In [78]:
# final_marker_annotation.to_parquet("../R_script_data/final_marker_annotation_CpG5_CV0.35_Top50_NoDis.parquet", engine="pyarrow")
# reference_map_df.to_parquet("../R_script_data/reference_map_df_CpG5_CV0.35_Top50_NoDis.parquet", engine="pyarrow")

# perform deconvolution

In [38]:
import os
import socket
import pandas as pd
import pyarrow
import numpy as np
from scipy.optimize import nnls

print(socket.gethostname())
os.chdir("/home/u24211510018/workspace/Atlas_WGBS/tissue_cell_type_methylation_density")
print(os.getcwd())

cpu10
/home/u24211510018/workspace/Atlas_WGBS/tissue_cell_type_methylation_density


In [39]:
# CpG=5, CV=0.35, Top50 NoDis Hyper and Hypo of Ref Map
reference_map_df = pd.read_parquet("../R_script_data/reference_map_df_CpG5_CV0.35_Top50_NoDis.parquet", engine="pyarrow")
final_marker_annotation = pd.read_parquet("../R_script_data/final_marker_annotation_CpG5_CV0.35_Top50_NoDis.parquet", engine="pyarrow")

In [40]:
final_marker_annotation.shape

(4138, 4)

In [41]:
reference_map_df.shape

(4138, 83)

In [4]:
# 按照 'tissue' 列分组，统计每个组合的行数
tissue_counts = final_marker_annotation.groupby('tissue').size()

# 显示结果
print(tissue_counts)

tissue
Adipocytes                           41
Adipocytes,Endometrium-Epithelial     1
Adipocytes,Pancreas-Endothel          1
Adipocytes,Prostate-Smooth-Muscle     2
Aorta-Endothel                       50
                                     ..
Thyroid-Epithelial                   50
Tongue-Epithelial                    50
Tongue_base-Epithelial               50
Tonsil-Palatine-Epithelial           50
Tonsil-Pharyngeal-Epithelial         50
Length: 89, dtype: int64


In [5]:
final_marker_annotation["cpg_count"].mean()

np.float64(19.153455775737072)

In [6]:
reference_map_df.head()

,Adipocytes,Aorta-Endothel,Aorta-Smooth-Muscle,Bladder-Epithelial,Bladder-Smooth-Muscle,Blood-B,Blood-B-Mem,Blood-Granulocytes,Blood-Monocytes,Blood-NK,...,Prostate-Smooth-Muscle,Saphenous-Vein-Endothel,Skeletal-Muscle,Small-int-Endocrine,Small-int-Epithelial,Thyroid-Epithelial,Tongue-Epithelial,Tongue_base-Epithelial,Tonsil-Palatine-Epithelial,Tonsil-Pharyngeal-Epithelial
chr18_79358000_79358500,67.405333,20.8330,27.083,29.0000,8.333,22.483333,15.2750,15.278000,26.666667,18.027667,...,11.117,12.039000,17.2250,11.900,24.98750,23.922000,19.283333,27.783,9.227667,34.3750
chr5_181053500_181054000,21.302000,5.2405,3.851,5.5152,4.864,6.222333,4.3410,4.790667,4.386000,5.502333,...,4.701,4.660000,2.9145,3.539,3.40050,3.505667,4.183333,3.838,3.517667,2.3180
chr8_127391500_127392000,16.136667,46.1000,31.464,46.8748,24.076,80.840000,81.1000,43.970000,52.857333,82.380000,...,27.952,37.666000,39.3490,94.880,92.00000,88.953333,35.123333,34.226,36.710667,39.3190
chr22_38317000_38317500,11.347667,2.3370,4.028,3.5728,2.410,4.122667,2.1875,3.014000,2.812667,3.956000,...,1.850,6.769333,3.0475,1.701,2.89375,1.205000,3.584000,3.333,2.414000,3.4275
chr1_153774500_153775000,13.373667,26.5775,24.642,41.6622,22.517,76.211000,79.6250,44.950667,56.403333,53.824000,...,23.938,25.038333,50.7345,52.332,51.48400,59.358667,28.226667,26.417,29.098000,36.3525


In [55]:
# using public data to test the rep map.
sample_files = {
    "ENCODE_Pancreas1": "/home/u24211510018/workspace/Atlas_WGBS/Pancreas/ENCFF080TDH_CpG_dyad.methylation_density.bed",
    "ENCODE_Pancreas2": "/home/u24211510018/workspace/Atlas_WGBS/Pancreas/ENCFF689HVV_CpG_dyad.methylation_density.bed",
    "breast_tumor": "/home/u24211510018/workspace/Atlas_WGBS/asTair_result/breast_tumor_human_CpG_dyad.methylation_density.bed",
}

In [56]:
def read_methylation_bed(file_path):
    df = pd.read_csv(
        file_path,
        sep="\t",
        header=None,
        names=["chr", "start", "end", "value"]
    )
    
    df["region_id"] = (
        df["chr"].astype(str) + "_" +
        df["start"].astype(str) + "_" +
        df["end"].astype(str)
    )
    
    s = pd.Series(df["value"].values, index=df["region_id"].values, name=os.path.basename(file_path))
    return s

In [57]:
sample_series_dict = {}

for sample_name, file_path in sample_files.items():
    sample_series_dict[sample_name] = read_methylation_bed(file_path)

for k, v in sample_series_dict.items():
    print(k, v.shape)
    print(v.head())

ENCODE_Pancreas1 (5008612,)
chr1_10000_10500    92.367
chr1_10500_11000    57.346
chr1_13000_13500    50.780
chr1_14500_15000    49.581
chr1_15000_15500    22.407
Name: ENCFF080TDH_CpG_dyad.methylation_density.bed, dtype: float64
ENCODE_Pancreas2 (5004946,)
chr1_10000_10500    67.217
chr1_10500_11000    33.683
chr1_13000_13500    57.500
chr1_14000_14500     0.000
chr1_14500_15000    53.125
Name: ENCFF689HVV_CpG_dyad.methylation_density.bed, dtype: float64
breast_tumor (4943206,)
chr1_10000_10500    95.233
chr1_10500_11000    17.073
chr1_12500_13000    14.286
chr1_13000_13500    62.780
chr1_13500_14000    33.510
Name: breast_tumor_human_CpG_dyad.methylation_density.bed, dtype: float64


In [58]:
def nnls_deconvolution(sample_series, reference_map_df, min_common_blocks=100):
    # 找共同 block
    common_blocks = reference_map_df.index.intersection(sample_series.index)
    
    if len(common_blocks) < min_common_blocks:
        raise ValueError(f"共同 block 太少: {len(common_blocks)}，小于阈值 {min_common_blocks}")
    
    # 参考矩阵 X：行为 block，列为 tissue
    X = reference_map_df.loc[common_blocks].values.astype(float)
    
    # 待分解样本 y：行为 block
    y = sample_series.loc[common_blocks].values.astype(float)
    
    # NNLS
    coef, residual = nnls(X, y)
    
    # 归一化为比例
    coef_sum = coef.sum()
    if coef_sum > 0:
        coef_prop = coef / coef_sum
    else:
        coef_prop = coef
    
    result_df = pd.DataFrame({
        "tissue": reference_map_df.columns,
        "coefficient": coef,
        "proportion": coef_prop
    }).sort_values("proportion", ascending=False).reset_index(drop=True)
    
    return result_df, residual, len(common_blocks)

In [59]:
pd.set_option('display.max_rows', None)
deconv_results = {}

for sample_name, sample_series in sample_series_dict.items():
    result_df, residual, n_blocks = nnls_deconvolution(
        sample_series=sample_series,
        reference_map_df=reference_map_df,
        min_common_blocks=100
    )
    
    deconv_results[sample_name] = {
        "result_df": result_df,
        "residual": residual,
        "n_common_blocks": n_blocks
    }
    
    print("=" * 60)
    print(sample_name)
    print("共同 block 数:", n_blocks)
    print("residual:", residual)
    print(result_df.head(83))

ENCODE_Pancreas1
共同 block 数: 4107
residual: 437.7591038289946
                                 tissue  coefficient  proportion
0                       Pancreas-Acinar     0.653705    0.670674
1                         Pancreas-Duct     0.206556    0.211918
2                     Pancreas-Endothel     0.032923    0.033778
3                        Megakaryocytes     0.030332    0.031119
4                   Aorta-Smooth-Muscle     0.024442    0.025077
5         Coronary-Artery-Smooth-Muscle     0.015831    0.016242
6          Kidney-Glomerular-Epithelial     0.003183    0.003266
7                         Pancreas-Beta     0.002719    0.002789
8                     Colon-Macrophages     0.002588    0.002655
9                     Cerebellum-Neuron     0.001403    0.001439
10              Pancreas-Islet-Endothel     0.000549    0.000563
11               Prostate-Smooth-Muscle     0.000396    0.000407
12               Endometrium-Epithelial     0.000072    0.000073
13                          